In [ ]:
"""
HorusEye Q1 niveau 2 — Trouver des candidats "personne au sol" dans RefCOCO
À lancer sur TA copie de RefCOCO (là où sont refs(unc).p et instances.json).

Objectif : présélectionner des images de personnes en posture NON-debout
(au sol, assise, penchée) pour que TU les annotes manuellement à la main,
comme tu l'as fait pour tes 50 premières images.

Le script NE crée PAS d'annotations — il liste des CANDIDATS à vérifier.
Tu regardes chaque image et tu confirmes/corriges la posture toi-même.
"""

import pickle
import json
import os
from collections import defaultdict

# ============================================================
# CONFIG — ajuste ces chemins vers ta copie de RefCOCO
# ============================================================
REFCOCO_DIR = "/Volumes/TheDay/thedayproject/Cours_Udem/cours_VL/HorusEye/Horus_dev/rq1_datasets/refcoco/refer/data/refcoco"              # dossier contenant refs(unc).p
INSTANCES = "/Volumes/TheDay/thedayproject/Cours_Udem/cours_VL/HorusEye/Horus_dev/rq1_datasets/refcoco/refer/data/refcoco/instances.json" # annotations COCO
COCO_IMAGES = "/Volumes/TheDay/thedayproject/Cours_Udem/cours_VL/HorusEye/Horus_dev/rq1_datasets/coco"       # images (pour vérifier existence)
OUTPUT = "./lying_candidates.json"
ALREADY_ANNOTATED = "/Volumes/TheDay/thedayproject/Cours_Udem/cours_VL/HorusEye/Horus_dev/RQ3_health_vqa/rq3_health_annotations.json"  # pour EXCLURE tes 50 déjà faites

# Mots-clés d'expressions indiquant une posture NON-debout
POSTURE_KEYWORDS = {
    "lying":    ["lying", "laying", "lie down", "lies", "reclining", "sprawled"],
    "on_ground":["on the ground", "on the floor", "on ground", "on floor",
                 "on the grass", "on the beach", "on the bed", "on a bed",
                 "on the couch", "on the sofa"],
    "sleeping": ["sleeping", "asleep", "napping", "passed out"],
    "sitting":  ["sitting", "seated", "sits", "crouching", "kneeling", "squatting"],
    "bent":     ["bending", "bent over", "leaning", "crouched", "hunched"],
}

MAX_PER_CATEGORY = 40  # combien de candidats max par catégorie à sortir

# ============================================================
# 1. CHARGER REFCOCO
# ============================================================
def load_refcoco():
    ref_file = os.path.join(REFCOCO_DIR, "refs(unc).p")
    if not os.path.exists(ref_file):
        print(f"⚠️  {ref_file} introuvable. Contenu de {REFCOCO_DIR}:")
        if os.path.isdir(REFCOCO_DIR):
            for f in os.listdir(REFCOCO_DIR): print("   ", f)
        raise SystemExit("Ajuste REFCOCO_DIR.")
    with open(ref_file, "rb") as f:
        refs = pickle.load(f)
    print(f"✓ {len(refs)} référents chargés")

    with open(INSTANCES) as f:
        inst = json.load(f)
    # bbox par annotation id
    ann_bbox = {a["id"]: a["bbox"] for a in inst["annotations"]}
    # dims image par image id
    img_info = {i["id"]: (i["width"], i["height"], i["file_name"])
                for i in inst["images"]}
    # ne garder que la catégorie "person"
    person_cat = [c["id"] for c in inst["categories"] if c["name"] == "person"]
    person_cat = person_cat[0] if person_cat else 1
    person_anns = {a["id"] for a in inst["annotations"] if a["category_id"] == person_cat}
    return refs, ann_bbox, img_info, person_anns

In [4]:
# ============================================================
# 2. FILTRER LES CANDIDATS
# ============================================================
def find_candidates():
    refs, ann_bbox, img_info, person_anns = load_refcoco()

    # exclure les images déjà annotées (tes 50)
    already = set()
    if os.path.exists(ALREADY_ANNOTATED):
        d = json.load(open(ALREADY_ANNOTATED))
        for s in d.get("samples", []):
            already.add(str(s.get("filename", "")).replace(".jpg", ""))
        print(f"✓ {len(already)} images déjà annotées seront exclues")

    candidates = defaultdict(list)
    seen = set()

    for ref in refs:
        ann_id = ref["ann_id"]
        # seulement les personnes
        if ann_id not in person_anns:
            continue
        # toutes les expressions de ce référent
        sentences = [s["sent"].lower() for s in ref["sentences"]]
        text = " ".join(sentences)

        # matcher une catégorie de posture
        matched_cat = None
        for cat, kws in POSTURE_KEYWORDS.items():
            if any(kw in text for kw in kws):
                matched_cat = cat
                break
        if not matched_cat:
            continue

        image_id = ref["image_id"]
        if image_id not in img_info:
            continue
        w, h, fname = img_info[image_id]
        base = fname.replace(".jpg", "").split("_")[-1]  # ex COCO_train2014_000000123 -> 000000123
        if base in already or base in seen:
            continue

        bbox = ann_bbox.get(ann_id)
        if not bbox:
            continue
        bw, bh = bbox[2], bbox[3]
        ratio = bw / bh if bh > 0 else 0

        candidates[matched_cat].append({
            "image_id": image_id,
            "file_name": fname,
            "ann_id": ann_id,
            "bbox": bbox,
            "bbox_ratio_wh": round(ratio, 2),
            "expressions": sentences[:3],
            "posture_hint": matched_cat,
            "posture_TO_ANNOTATE": ""   # ← TOI tu remplis après avoir regardé l'image
        })
        seen.add(base)

    return candidates

# ============================================================
# 3. SORTIE
# ============================================================
def main():
    cands = find_candidates()

    print("\n" + "=" * 55)
    print("CANDIDATS TROUVÉS (à vérifier visuellement)")
    print("=" * 55)
    total = 0
    output = []
    for cat, items in cands.items():
        # prioriser ceux dont la bbox est "paysage" (ratio>1) = plus probablement au sol
        items.sort(key=lambda x: -x["bbox_ratio_wh"])
        items = items[:MAX_PER_CATEGORY]
        print(f"\n  [{cat}] : {len(items)} candidats")
        for it in items[:3]:
            print(f"    {it['file_name']} | ratio={it['bbox_ratio_wh']} | \"{it['expressions'][0][:50]}\"")
        output.extend(items)
        total += len(items)

    with open(OUTPUT, "w") as f:
        json.dump({
            "note": "CANDIDATS à annoter manuellement. Regarde chaque image, "
                    "remplis 'posture_TO_ANNOTATE' avec STANDING/SITTING/LYING. "
                    "Le 'posture_hint' est une SUGGESTION basée sur le texte, PAS la vérité.",
            "candidates": output
        }, f, indent=2)

    print(f"\n✓ {total} candidats écrits dans {OUTPUT}")
    print("\nÉTAPE SUIVANTE (toi) :")
    print("  1. Ouvre chaque image candidate")
    print("  2. Regarde la VRAIE posture de la personne (bbox fournie)")
    print("  3. Remplis 'posture_TO_ANNOTATE' : STANDING / SITTING / LYING")
    print("  4. Garde ~15-20 vrais LYING pour équilibrer ton dataset")
    print("\n⚠️  Le 'posture_hint' vient du TEXTE, il peut être faux.")
    print("    Ex: 'sitting on the ground' peut être assis OU allongé.")
    print("    C'est TON œil sur l'image qui décide, pas le mot-clé.")

if __name__ == "__main__":
    main()

✓ 50000 référents chargés


FileNotFoundError: [Errno 2] No such file or directory: '/Volumes/TheDay/thedayproject/Cours_Udem/cours_VL/HorusEye /Horus_dev/refcoco_degraded_benchmark/annotations/annotations.json'